In [ ]:
import os
import random
import pandas as pd
import json
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

SEQ_LEN = 256
NUM_SAMPLES_PER_MODE = 2000

def get_texts(num_texts):
    print("Loading dataset from CSV...")
    try:
        df = pd.read_csv('news_summary.csv', encoding='latin-1')
        raw_texts = df['text'].dropna().head(num_texts).tolist()
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return []

    texts = []
    for text in raw_texts:
        text_bytes = text.encode('utf-8')
        if len(text_bytes) >= SEQ_LEN:
            texts.append(text_bytes[:SEQ_LEN])
        else:
            padded_bytes = text_bytes + b' ' * (SEQ_LEN - len(text_bytes))
            texts.append(padded_bytes)

    return texts

def encrypt_aes(plaintext, mode_name):
    key = os.urandom(32)
    iv = os.urandom(16)

    if mode_name == 'CFB':
        mode = modes.CFB(iv)
    elif mode_name == 'OFB':
        mode = modes.OFB(iv)
    elif mode_name == 'CTR':
        mode = modes.CTR(iv)
    else:
        raise ValueError("Invalid mode")

    cipher = Cipher(algorithms.AES(key), mode, backend=default_backend())
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(plaintext) + encryptor.finalize()
    return ciphertext

def generate_data():
    texts = get_texts(NUM_SAMPLES_PER_MODE * 3)

    if not texts:
        print("No texts loaded. Exiting...")
        return

    modes_list = ['CFB', 'OFB', 'CTR']
    data = []

    print("Generating ciphertexts...")
    for i, mode in enumerate(modes_list):
        for j in range(NUM_SAMPLES_PER_MODE):
            idx = i * NUM_SAMPLES_PER_MODE + j

            if idx >= len(texts):
                break

            plaintext = texts[idx]
            ciphertext = encrypt_aes(plaintext, mode)
            data.append({
                'ciphertext': list(ciphertext),
                'mode': mode
            })

    random.shuffle(data)

    with open('ciphertexts.json', 'w') as f:
        json.dump(data, f)
    print(f"Generated {len(data)} samples saved to ciphertexts.json")

if __name__ == "__main__":
    generate_data()

Loading dataset from CSV...
Generating ciphertexts...


/var/folders/_p/d06km2k91v3753mkftwkfnqm0000gn/T/ipykernel_54481/3001596949.py:41: CryptographyDeprecationWarning: CFB has been moved to cryptography.hazmat.decrepit.ciphers.modes.CFB and will be removed from cryptography.hazmat.primitives.ciphers.modes in 49.0.0.
  mode = modes.CFB(iv)
/var/folders/_p/d06km2k91v3753mkftwkfnqm0000gn/T/ipykernel_54481/3001596949.py:43: CryptographyDeprecationWarning: OFB has been moved to cryptography.hazmat.decrepit.ciphers.modes.OFB and will be removed from cryptography.hazmat.primitives.ciphers.modes in 49.0.0.
  mode = modes.OFB(iv)


Generated 4514 samples saved to ciphertexts.json
